In [25]:
# Cell 1 Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


In [26]:
# Step A2 Helper: Download Scikit-Learn 1.9.1 files & print Data Inventory table
import requests
import hashlib
import re

sources = [
    ("S01", "doc/modules/linear_model.rst", "https://scikit-learn.org/stable/modules/linear_model.html", "Supervised learning"),
    ("S02", "doc/modules/compose.rst", "https://scikit-learn.org/stable/modules/compose.html", "Pipelines and utilities"),
    ("S03", "doc/modules/cross_validation.rst", "https://scikit-learn.org/stable/modules/cross_validation.html", "Model selection and evaluation"),
    ("S04", "doc/modules/clustering.rst", "https://scikit-learn.org/stable/modules/clustering.html", "Unsupervised learning"),
    ("S05", "doc/modules/preprocessing.rst", "https://scikit-learn.org/stable/modules/preprocessing.html", "Dataset transformations"),
    ("S06", "doc/faq.rst", "https://scikit-learn.org/stable/faq.html", "General FAQ and usage"),
]

# Save official BSD-3-Clause licence
lic_url = "https://raw.githubusercontent.com/scikit-learn/scikit-learn/1.9.1/COPYING"
(BASE_DIR / "data" / "LICENSE-scikit-learn.txt").write_text(requests.get(lic_url).text, encoding="utf-8")

print("| Source ID | Filename / URL | Type and purpose | Version / date / owner | Access / rights | Inspection evidence |")
print("|---|---|---|---|---|---|")

for sid, repo_path, page_url, topic in sources:
    raw_url = f"https://raw.githubusercontent.com/scikit-learn/scikit-learn/1.9.1/{repo_path}"
    resp = requests.get(raw_url, timeout=15)
    resp.raise_for_status()
    content = resp.text

    # Save with .txt extension so Pathway A (Cells 5A1 & 8A) works natively
    local_name = repo_path.split("/")[-1].replace(".rst", ".txt")
    out_file = RAW_DIR / local_name
    out_file.write_text(content, encoding="utf-8")

    size_bytes = out_file.stat().st_size
    lines = len(content.splitlines())
    doctests = len(re.findall(r"^\s*>>>", content, flags=re.MULTILINE))
    sha_prefix = hashlib.sha256(out_file.read_bytes()).hexdigest()[:12]

    rights = "Public repository. BSD-3-Clause (licence copy in `data/LICENSE-scikit-learn.txt`). Raw file not committed." if sid == "S01" else "Same as S01"
    evidence = f"{size_bytes:,} bytes; {lines:,} lines; UTF-8; {doctests} doctest blocks. SHA-256 `{sha_prefix}…`."

    print(f"| {sid} | `{repo_path}`<br>{raw_url}<br>Page: {page_url} | Unstructured text (reStructuredText). Topic area: {topic}. | scikit-learn 1.9.1 (git tag); owner: scikit-learn developers; retrieved 2026-11-23 | {rights} | {evidence} |")

| Source ID | Filename / URL | Type and purpose | Version / date / owner | Access / rights | Inspection evidence |
|---|---|---|---|---|---|
| S01 | `doc/modules/linear_model.rst`<br>https://raw.githubusercontent.com/scikit-learn/scikit-learn/1.9.1/doc/modules/linear_model.rst<br>Page: https://scikit-learn.org/stable/modules/linear_model.html | Unstructured text (reStructuredText). Topic area: Supervised learning. | scikit-learn 1.9.1 (git tag); owner: scikit-learn developers; retrieved 2026-11-23 | Public repository. BSD-3-Clause (licence copy in `data/LICENSE-scikit-learn.txt`). Raw file not committed. | 86,036 bytes; 1,957 lines; UTF-8; 62 doctest blocks. SHA-256 `b9d050bf8b2c…`. |
| S02 | `doc/modules/compose.rst`<br>https://raw.githubusercontent.com/scikit-learn/scikit-learn/1.9.1/doc/modules/compose.rst<br>Page: https://scikit-learn.org/stable/modules/compose.html | Unstructured text (reStructuredText). Topic area: Pipelines and utilities. | scikit-learn 1.9.1 (git tag); owner: s

In [27]:
# Cell 3 Verify uploaded inputs (self-contained)
from pathlib import Path

RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 6
clustering.txt bytes: 94527
compose.txt bytes: 24753
cross_validation.txt bytes: 42081
faq.txt bytes: 28351
linear_model.txt bytes: 86036
preprocessing.txt bytes: 54344
Upload verified. Select your relevant input(s) for Step A3.


In [28]:
# Cell 4 REQUIRED: Source Selection + Lineage Utilities
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError("Select exactly one source. Set relative_path to an exact file under RAW_DIR.")
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return manifest

print("Cell 4 completed successfully.")

Cell 4 completed successfully.


In [29]:
# CELL 4A COMMON LINEAGE REGISTRY
from pathlib import Path

lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError("Replace the placeholder with the actual method.")
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print("Registered:", source.name, "->", output.name)

print("Common lineage registry ready.")

Common lineage registry ready.


In [30]:
# Cell 5A1 PATHWAY A1 - PDF/TXT
!pip -q install pypdf
from pypdf import PdfReader
import re

# Step 1: Select exact source file (S01: linear_model.txt)
f = select_source(relative_path="linear_model.txt")

# Step 2: Extract text
if f.suffix.lower() == ".pdf":
    reader = PdfReader(str(f))
    pages = [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]
elif f.suffix.lower() == ".txt":
    pages = [(1, f.read_text(encoding="utf-8"))]
else:
    raise ValueError("Pathway A needs a PDF or TXT Source")

# Step 3: Basic text cleaning (collapse horizontal spaces/tabs while keeping indentation/newlines)
text = "\n".join(content for _, content in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()

print("Source:", f.name)
print("Pages/segments:", len(pages))
print("Extracted characters:", len(text))
print("Cleaned characters:", len(clean))
print("Content sample:", repr(clean[:300]))
assert clean, "No usable text: check extraction or OCR needs."

# Step 4: Fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [s[i:i + size] for i in range(0, len(s), size)]

# Step 5: Paragraph-aware chunking
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))
        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts

# Step 6: Compare two preparation approaches
fixed = fixed_chunks(clean, size=500)
structured = paragraph_chunks(clean, size=500)

print("\n--- Chunking Comparison ---")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))
print("\nFixed sample:")
print(repr(fixed[0][:180]))
print("\nParagraph-aware sample:")
print(repr(structured[0][:180]))
print("\nPATHWAY A1 completed successfully.")

Source: linear_model.txt
Pages/segments: 1
Extracted characters: 86023
Cleaned characters: 82897
Content sample: '.. _linear_model:\n\n=============\nLinear Models\n=============\n\n.. currentmodule:: sklearn.linear_model\n\nThe following are a set of methods intended for regression in which\nthe target value is expected to be a linear combination of the features.\nIn mathematical notation, the predicted value :math:`\\ha'

--- Chunking Comparison ---
Fixed chunks: 166
Paragraph-aware chunks: 225

Fixed sample:
'.. _linear_model:\n\n=============\nLinear Models\n=============\n\n.. currentmodule:: sklearn.linear_model\n\nThe following are a set of methods intended for regression in which\nthe targe'

Paragraph-aware sample:
'.. _linear_model:\n\n=============\nLinear Models\n=============\n\n.. currentmodule:: sklearn.linear_model\n\nThe following are a set of methods intended for regression in which\nthe targe'

PATHWAY A1 completed successfully.


In [31]:
# Cell 5A2 PATHWAY A2 - Export traceable text chunks
import json

CHOSEN_METHOD = "paragraph-aware"
if CHOSEN_METHOD == "paragraph-aware":
    chosen = structured
    approach = "paragraph-aware-char-500"
elif CHOSEN_METHOD == "fixed":
    chosen = fixed
    approach = "fixed-char-500"
else:
    raise ValueError("Choose 'paragraph-aware' or 'fixed'")

records = [
    {
        "record_id": f"{f.stem}-{i:04d}",
        "text": chunk,
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_hash(f),
        "processing": approach,
        "page": None
    }
    for i, chunk in enumerate(chosen, 1)
]

prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")

assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), "One or more chunks are empty"

print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])

PATHWAY A2 completed successfully.
Selected approach: paragraph-aware-char-500
Total chunks exported: 225
Prepared file: /content/week3_project/data/processed/prepared_text.jsonl
File size: 133432 bytes
Validation: PASS
First record: {'record_id': 'linear_model-0001', 'text': '.. _linear_model:\n\n=============\nLinear Models\n=============\n\n.. currentmodule:: sklearn.linear_model\n\nThe following are a set of methods intended for regression in which\nthe target value is expected to be a linear combination of the features.\nIn mathematical notation, the predicted value :math:`\\hat{y}` can be\nwritten as:\n\n.. math:: \\hat{y}(w, x) = w_0 + w_1 x_1 + ... + w_p x_p', 'source': 'linear_model.txt', 'source_sha256': 'b9d050bf8b2ca47fb0b305255631629ef410ccf2944f198c7297f4e7b96266dc', 'processing': 'paragraph-aware-char-500', 'page': None}


In [32]:
# B3 Basic Output Check + B4.1 Register Pathway A + 6A & 6B Manifest
assert prepared_path.exists(), "Prepared output missing."
assert prepared_path.stat().st_size > 0, "Prepared output is empty."
print("B3 Output:", prepared_path, "Size:", prepared_path.stat().st_size)

# B4.1 Register Pathway A
register_lineage(source=f, output=prepared_path, processing=approach)

# Cell 6A Generate Common Lineage Manifest
assert lineage_entries, "No lineage entries registered."
lineage_records = []
for i, entry in enumerate(lineage_entries.values(), 1):
    src = entry["source"]
    out = entry["output"]
    proc = entry["processing"]
    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(src.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(src),
        "prepared_output": str(out.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(out),
        "processing_note": proc,
        "source_version": "scikit-learn 1.9.1",
        "access_rights": "BSD-3-Clause (public repository)",
        "status": "Prepared; quality review pending"
    })

manifest = save_manifest(lineage_records)

# Cell 6B Verify Common Lineage Manifest
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
saved_manifest = json.load(manifest_path.open("r", encoding="utf-8"))
saved_records = saved_manifest.get("records", [])
assert len(saved_records) == len(lineage_entries)
for record in saved_records:
    src = RAW_DIR / record["source_filename"]
    out = BASE_DIR / record["prepared_output"]
    assert source_hash(src) == record["source_sha256"]
    assert source_hash(out) == record["prepared_sha256"]

print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))

B3 Output: /content/week3_project/data/processed/prepared_text.jsonl Size: 133432
Registered: linear_model.txt -> prepared_text.jsonl
Saved lineage: /content/week3_project/data/week3_manifest.json
COMMON LINEAGE VALIDATION: PASS
Verified records: 1


In [33]:
# Cell 7 REQUIRED: Common V1 Pipeline Utilities
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_RUNS = {}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if folder:
        if not source.is_dir():
            raise FileNotFoundError(f"Source folder not found: {source}")
    else:
        if not source.is_file():
            raise FileNotFoundError(f"Source file not found: {source}")
    return source

def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    source = Path(source)
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
        )
    else:
        inputs = [source]
    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")

    inspection = inspect_fn(inputs, configuration)
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]

    if not outputs:
        raise ValueError("No prepared outputs were produced.")
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(f"Prepared output missing: {output}")
        if output.stat().st_size == 0:
            raise ValueError(f"Prepared output is empty: {output}")

    validation = {
        "output_count": len(outputs),
        "output_sha256": [sha256_file(p) for p in outputs],
        "output_bytes": [p.stat().st_size for p in outputs]
    }

    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [{"path": str(p), "sha256": sha256_file(p)} for p in inputs],
        "outputs": [str(p) for p in outputs],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "scikit-learn 1.9.1",
        "access_rights": "BSD-3-Clause (public repository)"
    }

    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")

    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    print("Manifest:", manifest_path)
    return record

def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    def runner():
        return execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first

print("Cell 7 ready.")

Cell 7 ready.


In [34]:
# Cell 8A CONDITIONAL REQUIRED: PDF/TXT Projects
import re
import json
from pypdf import PdfReader

SOURCE_A = resolve_source("linear_model.txt")
CONFIG_A = {
    "chunk_size": 500,
    "method": "paragraph-aware"
}

def inspect_documents(inputs, cfg):
    src = inputs[0]
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        pages = [page.extract_text() or "" for page in reader.pages]
        text = "\n".join(pages)
        page_count = len(pages)
    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")
        page_count = None
    else:
        raise ValueError("Expected a PDF or TXT source")
    if not text.strip():
        raise ValueError("No extractable text found.")
    return {
        "source": src.name,
        "characters": len(text),
        "pages": page_count
    }

def prepare_documents(inputs, out, cfg):
    src = inputs[0]
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")
    else:
        raise ValueError("Expected a PDF or TXT source")

    text = re.sub(r"[ \t]+", " ", text).strip()
    size = int(cfg["chunk_size"])
    method = cfg["method"]

    if method == "fixed":
        chunks = [text[i:i + size] for i in range(0, len(text), size)]
    elif method == "paragraph-aware":
        chunks = []
        current = ""
        for section in re.split(r"\n\s*\n", text):
            section = section.strip()
            if not segment if False else not section:
                continue
            if len(section) > size:
                if current:
                    chunks.append(current)
                    current = ""
                chunks.extend(section[i:i + size] for i in range(0, len(section), size))
            elif len(current) + len(section) + (2 if current else 0) <= size:
                current += ("\n\n" if current else "") + section
            else:
                if current:
                    chunks.append(current)
                current = section
        if current:
            chunks.append(current)
    else:
        raise ValueError("Choose 'fixed' or 'paragraph-aware'")

    target = out / "prepared_text.jsonl"
    with target.open("w", encoding="utf-8") as handle:
        for i, chunk in enumerate(chunks, 1):
            record = {
                "id": i,
                "source": str(src),
                "text": chunk,
                "page": None
            }
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    mapping = [{"source": str(src), "output": str(target)}]
    processing_method = f"{method} text chunks of {size} characters"
    return [target], mapping, processing_method

run_A = register_and_run("documents", SOURCE_A, inspect_documents, prepare_documents, CONFIG_A)
print("Cell 8A completed successfully.")

SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
Cell 8A completed successfully.


In [35]:
# Generate docs/week3_data_inventory.md and docs/week3_pipeline_decisions.md in Colab
DOCS_DIR = BASE_DIR / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

inventory_md = """# Week 3 Data Inventory — SklearnDocs Assistant

## Step A1: From Topic 2 requirements to data needs

| Requirement ID | Needed information / feature | Candidate source | Why usable / constraint |
|---|---|---|---|
| FR1 / FR2: answer from indexed documentation and give a source reference | Documentation text, module topic, and public page URL per record | Official `scikit-learn` documentation source files (`.rst` saved as `.txt` for Pathway A) from GitHub tag `1.9.1` | Official, public, and matches the live website wording. Constraint: citations must point to the public HTML documentation URL rather than the raw source file. |
| FR3: state the documentation version | Version label stored with every record | Same files; fixed at git tag `1.9.1` | Prevents drift when the website `stable` alias updates to newer releases. |
| RAI4: no unsupported class, function, or parameter names | Intact code blocks (doctest examples) and mathematical notation | Same files (602 total doctest blocks across S01–S06) | Fixed-size character slicing breaks code examples and formulas in half; paragraph-aware chunking is selected to preserve boundaries. |
| RAI1: similar coverage across topic areas | Representative source files spanning core Scikit-Learn domains plus the FAQ | Six official module files (`S01`–`S06`) covering supervised learning, pipelines, model selection, clustering, preprocessing, and FAQ | V1 ingests six core documents to establish and verify the pipeline before full corpus scaling. |

**Checkpoint A:** `FR1`, `FR2`, `FR3`, `RAI1`, and `RAI4` motivate these six sources. Inspection evidence is recorded in `notebooks/week3_pipeline.ipynb` (Step A2/A3), verifying file sizes, line counts, doctest counts, and extracted text samples. No dummy dataset was used.

## Step A4: Data inventory

| Source ID | Filename / URL | Type and purpose | Version / date / owner | Access / rights | Inspection evidence |
|---|---|---|---|---|---|
| S01 | `doc/modules/linear_model.rst` (saved as `linear_model.txt`)<br>https://raw.githubusercontent.com/scikit-learn/scikit-learn/1.9.1/doc/modules/linear_model.rst<br>Page: https://scikit-learn.org/stable/modules/linear_model.html | Unstructured text (reStructuredText). Topic area: Supervised learning. | scikit-learn 1.9.1 (git tag); owner: scikit-learn developers; retrieved 2026-11-23 | Public repository. BSD-3-Clause (licence copy in `data/LICENSE-scikit-learn.txt`). Raw file not committed. | 86,036 bytes; 1,957 lines; UTF-8; 62 doctest blocks. SHA-256 `b9d050bf8b2c…`. Prepared: 225 records (`prepared_text.jsonl`, 133,432 bytes). |
| S02 | `doc/modules/compose.rst` (saved as `compose.txt`)<br>https://raw.githubusercontent.com/scikit-learn/scikit-learn/1.9.1/doc/modules/compose.rst<br>Page: https://scikit-learn.org/stable/modules/compose.html | Unstructured text (reStructuredText). Topic area: Pipelines and utilities. | scikit-learn 1.9.1 (git tag); owner: scikit-learn developers; retrieved 2026-11-23 | Same as S01 | 24,753 bytes; 615 lines; UTF-8; 121 doctest blocks. SHA-256 `d116a5d4852c…`. |
| S03 | `doc/modules/cross_validation.rst` (saved as `cross_validation.txt`)<br>https://raw.githubusercontent.com/scikit-learn/scikit-learn/1.9.1/doc/modules/cross_validation.rst<br>Page: https://scikit-learn.org/stable/modules/cross_validation.html | Unstructured text (reStructuredText). Topic area: Model selection and evaluation. | scikit-learn 1.9.1 (git tag); owner: scikit-learn developers; retrieved 2026-11-23 | Same as S01 | 42,081 bytes; 1,026 lines; UTF-8; 128 doctest blocks. SHA-256 `06f22c6b86b0…`. |
| S04 | `doc/modules/clustering.rst` (saved as `clustering.txt`)<br>https://raw.githubusercontent.com/scikit-learn/scikit-learn/1.9.1/doc/modules/clustering.rst<br>Page: https://scikit-learn.org/stable/modules/clustering.html | Unstructured text (reStructuredText). Topic area: Unsupervised learning. | scikit-learn 1.9.1 (git tag); owner: scikit-learn developers; retrieved 2026-11-23 | Same as S01 | 94,527 bytes; 2,223 lines; UTF-8; 92 doctest blocks. SHA-256 `adf31a738dba…`. |
| S05 | `doc/modules/preprocessing.rst` (saved as `preprocessing.txt`)<br>https://raw.githubusercontent.com/scikit-learn/scikit-learn/1.9.1/doc/modules/preprocessing.rst<br>Page: https://scikit-learn.org/stable/modules/preprocessing.html | Unstructured text (reStructuredText). Topic area: Dataset transformations. | scikit-learn 1.9.1 (git tag); owner: scikit-learn developers; retrieved 2026-11-23 | Same as S01 | 54,344 bytes; 1,312 lines; UTF-8; 189 doctest blocks. SHA-256 `5f187d90bf28…`. |
| S06 | `doc/faq.rst` (saved as `faq.txt`)<br>https://raw.githubusercontent.com/scikit-learn/scikit-learn/1.9.1/doc/faq.rst<br>Page: https://scikit-learn.org/stable/faq.html | Unstructured text (reStructuredText). Topic area: General FAQ and usage. | scikit-learn 1.9.1 (git tag); owner: scikit-learn developers; retrieved 2026-11-23 | Same as S01 | 28,351 bytes; 597 lines; UTF-8; 10 doctest blocks. SHA-256 `cc19d3e204ae…`. |
"""

decisions_md = """# Week 3 Pipeline Decisions — SklearnDocs Assistant

## 1. Traceability & Selected Source
- **Linked Topic 2 Requirements:** `FR1` (documentation-grounded answers), `FR2` (source citation), `FR3` (explicit version tracking `1.9.1`), `RAI1` (balanced topic coverage), and `RAI4` (preserving exact code/parameter syntax).
- **Selected Pathway:** Pathway A (Unstructured Text — `.txt` files fetched from official `scikit-learn` v1.9.1 `.rst` documentation sources).
- **Primary Pipeline Source:** `linear_model.txt` (`S01`, 86,036 bytes, SHA-256 `b9d050bf8b2ca47fb0b305255631629ef410ccf2944f198c7297f4e7b96266dc`).

## 2. Part B Preparation Comparison & Evidence

| Approach A | Approach B | Observed Evidence (`linear_model.txt`) | Selected Approach & Trade-off |
|---|---|---|---|
| **Fixed-size chunking** (`size=500` characters) | **Paragraph-aware chunking** (`size=500` characters, splitting on `\\n\\s*\\n`) | Raw characters: `86,023` → Cleaned characters: `82,897`.<br>- **Fixed-size:** `166` chunks; cuts mid-sentence and splits LaTeX/doctest blocks across boundaries.<br>- **Paragraph-aware:** `225` chunks (`133,432` bytes exported); preserves complete paragraphs and math definitions (e.g., record `linear_model-0001` keeps the entire Linear Models intro and `\\hat{y}` equation intact). | **Selected: Paragraph-aware (`paragraph-aware-char-500`).**<br>**Benefit:** Directly supports `RAI4` and `FR1` by preventing arbitrary mid-sentence/mid-equation splits.<br>**Accepted Limitation:** Produces more total chunks (`225` vs `166`), and any single paragraph or code block exceeding 500 characters falls back to fixed character slicing. |

## 3. Architecture & Engineering Trade-offs
- **ETL vs. ELT:** **ETL (Extract, Transform, Load)** is used. Raw documentation is extracted from GitHub, cleaned (horizontal whitespace normalization) and segmented into structured JSONL records (`record_id`, `text`, `source`, `source_sha256`, `processing`) before saving to processed storage.
- **Batch vs. Streaming:** **Batch processing** is selected because `scikit-learn` releases versioned documentation snapshots periodically (e.g., tag `1.9.1`) rather than as continuous real-time event streams.
- **Storage & Provenance Caveat:** Raw files are held in temporary Colab storage (`data/raw/`) during execution and excluded from Git commits to keep the repository lightweight; full reproducibility is preserved via pinned GitHub `1.9.1` URLs, `data/LICENSE-scikit-learn.txt`, `data/week3_manifest.json`, and `data/pipeline_v1/documents/pipeline_manifest.json`.

## 4. Stage Contracts & Failure Responses

| Stage | Contract / Evidence | Failure Response |
|---|---|---|
| **Ingest** | Selected source file exists in `data/raw/` via explicit `resolve_source()` / `select_source()`. | Raise `FileNotFoundError` and halt execution immediately. |
| **Inspect** | File decodes as valid UTF-8 and contains non-empty readable characters (`86,023` chars). | Raise `ValueError` / `AssertionError` if extracted text is empty. |
| **Prepare** | Whitespace normalization and paragraph-aware chunking (`500` chars) produce non-empty text segments. | Raise `ValueError` if chunk size `<= 0` or zero chunks are produced. |
| **Validate** | Output `prepared_text.jsonl` exists on disk, has byte size `> 0` (`133,432` bytes), and passes repeatability SHA-256 smoke checks (Cell 9). | Raise `AssertionError` / `FileNotFoundError` if output is missing, empty, or non-deterministic across reruns. |
| **Record Lineage** | `week3_manifest.json` and `pipeline_manifest.json` map exact source SHA-256 to output SHA-256 and processing metadata. | Raise `AssertionError` on checksum mismatch or missing lineage entries. |
"""

(DOCS_DIR / "week3_data_inventory.md").write_text(inventory_md, encoding="utf-8")
(DOCS_DIR / "week3_pipeline_decisions.md").write_text(decisions_md, encoding="utf-8")
print("Created both markdown files in:", DOCS_DIR)

Created both markdown files in: /content/week3_project/docs
